**<h3 style="color:orange">Please make sure any LLM coding assistant like Github Copilot or Cursus is turned OFF!</h2>**

**<h4 style="color:orange">We want you to learn and think critically, not to let LLMs do your work for you.</h4>**

**<h4 style="color:orange">Ask your group members or the internet for help, BEFORE asking your favorite chatbot.</h4>**

<img src="../figures/copilot.png">

# Assignment 1 Part 2: Modelling Boids as a Lagrangian System


In [ ]:
# Import all the required libraries here
import torch as T
from torch import nn
from torch.utils.data import Dataset
from torch_geometric.data import Data, DataLoader as GraphDataLoader
import lightning as L
import os
import pickle


print(f"Success! Running PyTorch {T.__version__} and Lightning {L.__version__}")
cuda_available = T.cuda.is_available()
if cuda_available:
    print(f"CUDA is available! Device count: {T.cuda.device_count()}")
else:
    print("CUDA is not available.")

In [ ]:
class BoidsDataset(Dataset):
    """Dataset for the Boids simulation data."""

    def __init__(self, path):
        self.path = path
        self.nodes = T.tensor([], dtype=T.float32)  
        self.edge_index = T.tensor([], dtype=T.long)
        self.edge_attributes = T.tensor([], dtype=T.float32) 
        self.n_simulations = 0
        self.traj_length = 0

    def load_data(self):
        if not os.path.exists(self.path):
            raise FileNotFoundError(
                f"Data file not found at {self.path}. Please check the path and try again."
            )
        with open(self.path, "rb") as f:
            data = pickle.load(f)

        # TODO: Implement the logic to load nodes and edges from the data dictionary

        print(
            f"\nLoaded data from {self.path}, containing {len(data)} samples of shape {data[0].shape}\n"
        )

class BoidsTrajectoryDataset(BoidsDataset):
    """One sample = one full trajectory: nodes has shape (traj_len, ...)."""

    def __len__(self):
        return self.n_simulations

    def __getitem__(self, idx):
        return Data(
            x=self.nodes[idx],
            edge_index=self.edge_index[idx],
            edge_attr=self.edge_attributes[idx]
        )


class BoidsStepDataset(BoidsDataset):
    """One sample = one timestep: nodes has shape (...)."""

    def __len__(self):
        return self.n_simulations * self.traj_length

    def __getitem__(self, idx):
        sim_idx, t_idx = idx // self.traj_length, idx % self.traj_length
        return Data(
            x=self.nodes[sim_idx][t_idx],
            edge_index=self.edge_index[sim_idx][t_idx],
            edge_attr=self.edge_attributes[sim_idx][t_idx]
        )

class DataModule(L.LightningDataModule):
    """Lightning DataModule class that manages the dataset splitting and the dataloaders"""

    def __init__(self, dataset, batch_size, train_split, val_split, seed=42):
        super().__init__()
        self.dataset = dataset
        self.batch_size = batch_size
        self.train_split = train_split
        self.val_split = val_split
        self.seed = seed

    def prepare_data(self):
        self.dataset.load_data()

    def setup(self, stage=None):
            if getattr(self, "train_set", None) is not None:
                return

            # TODO: Implement logic to split the dataset into train, validation, and test sets

    def train_dataloader(self):
        return GraphDataLoader(self.train_set, batch_size=self.batch_size)

    def val_dataloader(self):
        return GraphDataLoader(self.val_set, batch_size=self.batch_size)

    def test_dataloader(self):
        return GraphDataLoader(self.test_set, batch_size=self.batch_size)

    def predict_dataloader(self):
        return GraphDataLoader(self.test_set, batch_size=self.batch_size)

path = os.path.join(os.getcwd(), "..", "data", "boids","boids_dataset.pkl")
boids_dataset = BoidsTrajectoryDataset(path=path)
boids_dataset.load_data()

In [ ]:
# Perform the exploratory data analysis here

In [ ]:
class EGNN(nn.Module):

    def __init__(self):
        super().__init__()
        # TODO: Initialize the EGNN architecture here

    def forward(self, x, edge_index, edge_attr):

        # TODO: Implement the forward pass of the EGNN here

        return x

In [ ]:
class BoidsModel(L.LightningModule):
    def __init__(self, model, learning_rate=1e-3):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def _step(self, batch):

        # TODO Implement the logic for a single training/validation step here
        
        return batch

    def training_step(self, batch):
        loss = self._step(batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch):
        loss = self._step(batch)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def predict_step(self, batch):

        # TODO Implement the logic for autoregressive rollout here

        return batch

    def configure_optimizers(self):
        return T.optim.Adam(self.parameters(), lr=self.learning_rate)

class LossHistory(L.Callback):
    def __init__(self):
        self.train_loss, self.val_loss = [], []
    def on_train_epoch_end(self, trainer, pl_module):
        self.train_loss.append(trainer.callback_metrics.get("train_loss").item())
    def on_validation_epoch_end(self, trainer, pl_module):
        self.val_loss.append(trainer.callback_metrics.get("val_loss").item())

In [ ]:
# Initialize dataset
dataset = BoidsStepDataset(path)

# Initialize datamodule
datamodule = DataModule(dataset)

# Initialize model
egnn = EGNN()
model = BoidsModel(egnn)

# Initialize trainer
loss_history = LossHistory()
trainer = L.Trainer(callbacks=[loss_history])

# Fit the model to the dataset
trainer.fit(model, datamodule=datamodule)

# Load the inference dataset 
traj_dataset = BoidsTrajectoryDataset(path)
datamodule = DataModule(traj_dataset)

# Perform inference
rollouts = trainer.predict(model, datamodule=datamodule)

In [ ]:
# Evaluate the model and make plots here